# CIND820 Milestone 3 Proof-of-Concept Pipeline Demo

This compact notebook demonstrates that the project pipeline is functional from **data ingestion** through **preprocessing**, **chronological validation**, and **initial model evaluation**.

It uses the same methodology and parameter choices as the main project notebooks, but calls reusable functions from `src/co2_pipeline.py` so the workflow is easier to reproduce and demonstrate live.

In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path("../src").resolve()))

from co2_pipeline import (
    load_emissions_data,
    assess_modelling_periods,
    build_forecasting_dataset,
    chronological_split,
    split_summary,
    evaluate_initial_models,
)

DATA_PATH = Path("../data/raw/GCB2022v27_MtCO2_flat.csv")
print("Pipeline module imported successfully.")
print("Data path:", DATA_PATH)

Pipeline module imported successfully.
Data path: ../data/raw/GCB2022v27_MtCO2_flat.csv


## 1. Data ingestion

The first proof point is that the authoritative GCB2022v27 flat file can be loaded with the expected schema and dimensions.

In [2]:
df = load_emissions_data(DATA_PATH)

print("Raw dataset shape:", df.shape)
print("Year range:", df["Year"].min(), "to", df["Year"].max())
print("Unique country/entity series:", df["Country"].nunique())

df.head()

Raw dataset shape: (63104, 11)
Year range: 1750 to 2021
Unique country/entity series: 232


       Country ISO 3166-1 alpha-3  Year  ...  Flaring  Other  Per Capita
0  Afghanistan                AFG  1750  ...      NaN    NaN         NaN
1  Afghanistan                AFG  1751  ...      NaN    NaN         NaN
2  Afghanistan                AFG  1752  ...      NaN    NaN         NaN
3  Afghanistan                AFG  1753  ...      NaN    NaN         NaN
4  Afghanistan                AFG  1754  ...      NaN    NaN         NaN

[5 rows x 11 columns]

## 2. Data-quality assessment and modelling-period choice

Candidate periods are compared using the same profiling measures used in the main analysis: core-variable completeness and the proportion of zero Total-emissions observations.

In [3]:
period_assessment = assess_modelling_periods(df)
period_assessment

   Start Year   Rows  Core Complete Rows  Core Complete %  Zero Total %
0        1900  28304               15320             54.1          32.8
1        1950  16704               14747             88.3          10.2
2        1960  14384               12956             90.1           8.3
3        1980   9744                8904             91.4           6.4
4        1990   7424                6848             92.2           5.0
5        2000   5104                4743             92.9           4.8

**POC interpretation:** 1960 provides approximately 90.1% completeness across the core modelling variables and reduces zero Total observations to about 8.3%, while retaining substantially more historical data than later start years.

## 3. Forecasting dataset construction

This step applies the project population rule, creates the within-entity `t+1` target, assigns the year-`t` predictors, checks the one-year forecast gap, and keeps complete predictor rows.

In [4]:
model_df, diagnostics = build_forecasting_dataset(df, start_year=1960)

diagnostics

{'start_year': 1960,
 'excluded_entities': ['French Equatorial Africa', 'French West Africa', 'Global', 'International Transport', 'Kuwaiti Oil Fires', 'Leeward Islands', 'Pacific Islands (Palau)', 'Ryukyu Islands'],
 'excluded_entity_count': 8,
 'provisional_rows': 13888,
 'provisional_entities': 224,
 'valid_target_rows': 13664,
 'complete_case_rows': 12632,
 'complete_case_entities': 223,
 'predictor_year_min': 1960,
 'predictor_year_max': 2020}

In [5]:
model_df.loc[
    model_df["Country"].eq("Canada"),
    ["Country", "Year", "Total_lag1", "Target_Year", "Target_Total_next_year"]
].tail(5)

     Country  Year  Total_lag1  Target_Year  Target_Total_next_year
2288  Canada  2016  560.525313       2017.0              571.544619
2289  Canada  2017  571.544619       2018.0              584.369116
2290  Canada  2018  584.369116       2019.0              584.714184
2291  Canada  2019  584.714184       2020.0              534.863824
2292  Canada  2020  534.863824       2021.0              545.634517

The Canada example makes the forecasting logic visible: information from predictor year `t` is used to forecast the same entity's Total emissions in `t+1`.

## 4. Chronological validation protocol

No random shuffling is used because the task is time-ordered forecasting. Earlier years are used for training, later years for validation, and the most recent period is reserved for testing.

In [6]:
train_df, validation_df, test_df = chronological_split(model_df)
split_summary(train_df, validation_df, test_df)

        Split  Predictor Start Year  Predictor End Year   Rows  Entities
0    Training                  1960                2014  11329       223
1  Validation                  2015                2017    652       218
2        Test                  2018                2020    651       217

## 5. Initial model evaluation

The proof of concept compares a persistence baseline with Linear Regression, Random Forest Regression, and Gradient Boosting Regression on the initial validation window. The goal at Milestone 3 is to show that the pipeline generates interpretable model outputs, not to claim a final champion model.

In [7]:
initial_results, models = evaluate_initial_models(train_df, validation_df)
initial_results.round(3)

                  Model    MAE    RMSE     R2
0  Persistence Baseline  4.806  21.715  0.999
1     Linear Regression  5.166  27.404  0.999
2         Random Forest  5.354  26.724  0.999
3     Gradient Boosting  5.734  26.621  0.999

## 6. Initial finding and next step

The initial validation window shows that **persistence is the strongest benchmark** on MAE and RMSE. This proves the end-to-end pipeline is functioning and also reveals an important modelling challenge: year-to-year emissions persistence is difficult to outperform. The appropriate next step is broader chronological validation before any final model selection.